In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6JhhNjzvJvJOh7SNk0yOtnZUHvLnNEmpqL_junkn0EwMg"

Install Libraries

In [ ]:
! pip install -q youtube-transcript-api langchain-community langchain-google-genai \ faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.4 MB/s eta 0:00:00


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

Step 1(a): Indexing (Document Ingestion)

In [ ]:
video_id = "J5_-l7WIO_w"

ytt_api = YouTubeTranscriptApi()

transcript = ytt_api.fetch(video_id, languages=['hi'])

text = " ".join([snippet.text for snippet in transcript])

print(text)

हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अह पिछले वीडियो में हमने रैग पढ़ना शुरू किया था और हमने फोकस किया था रैग के अराउंड जो भी थ्योरी है उसको डिस्कस करने के ऊपर। मैंने आपको बताया था कि रैग क्या होता है? उसकी जरूरत क्यों होती है? मैंने वहां पर रैक को कंपेयर करके भी दिखाया था फाइन ट्यूनिंग जैसी टेक्निक के साथ। और आज का जो वीडियो है वह पिछले वीडियो का ही कंटिन्यूएशन है। जहां पर हम प्रैक्टिकली एक रैग बेस्ड सिस्टम बनाएंगे यूजिंग लैंग चेन। प्लान यह है कि मैं एक प्रॉब्लम स्टेटमेंट उठाऊंगा और उस प्रॉब्लम स्टेटमेंट के अराउंड एक रैग बेस्ड सिस्टम क्रिएट करूंगा और यह सारा का सारा कोड हम लैंग चेन में करने वाले हैं। तो अभी तक आपने जो भी पढ़ा है पिछले चारप वीडियोस में डॉक्यूमेंट लोडर्स, टेक्स्ट स्प्लिटर्स, वेक्टर स्टर्स ये सब कुछ हम आज के वीडियो में यूज़ करेंगे और इनको यूज़ करके हम एक रैग बेस सिस्टम बनाएंगे। ऑन द होल इट्स गोइंग टू बी अ वेरी इंटरेस्टिंग वीडियो। लेट्स स्टार्ट। तो चलो गाइस, सबसे पहले बात करते हैं प्र

1(b): Indexing (Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([text])

In [ ]:
len(chunks)

51

In [ ]:
chunks[0]

Document(metadata={}, page_content='हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अह पिछले वीडियो में हमने रैग पढ़ना शुरू किया था और हमने फोकस किया था रैग के अराउंड जो भी थ्योरी है उसको डिस्कस करने के ऊपर। मैंने आपको बताया था कि रैग क्या होता है? उसकी जरूरत क्यों होती है? मैंने वहां पर रैक को कंपेयर करके भी दिखाया था फाइन ट्यूनिंग जैसी टेक्निक के साथ। और आज का जो वीडियो है वह पिछले वीडियो का ही कंटिन्यूएशन है। जहां पर हम प्रैक्टिकली एक रैग बेस्ड सिस्टम बनाएंगे यूजिंग लैंग चेन। प्लान यह है कि मैं एक प्रॉब्लम स्टेटमेंट उठाऊंगा और उस प्रॉब्लम स्टेटमेंट के अराउंड एक रैग बेस्ड सिस्टम क्रिएट करूंगा और यह सारा का सारा कोड हम लैंग चेन में करने वाले हैं। तो अभी तक आपने जो भी पढ़ा है पिछले चारप वीडियोस में डॉक्यूमेंट लोडर्स, टेक्स्ट स्प्लिटर्स, वेक्टर स्टर्स ये सब कुछ हम आज के वीडियो में यूज़ करेंगे और इनको यूज़ करके हम एक रैग बेस सिस्टम बनाएंगे। ऑन द होल इट्स गोइंग टू बी अ वेरी इंटरेस्टिंग वीडियो। लेट्स स्टार्ट। तो 

In [ ]:
chunks[50]

Document(metadata={}, page_content='कब पढ़ाओगे? तो मैं बस क्लेरिफाई करना चाहता हूं कि ये जो टेक्निक्स मैंने आपको अभी 10-15 मिनट में बताई हैं ये हम लोग लैंग चेन वाले प्लेलिस्ट में कंटिन्यू कवर नहीं करेंगे। मेरा प्लान ये है कि एक बार जब ये लैंगचेन वाला प्लेलिस्ट कवर हो जाएगा उसके बाद मैं एक सेपरेट प्लेलिस्ट बनाऊंगा एडवांस्ड रैग बोल करके। वहां पर हम ये सारी चीजें करेंगे जो मैंने आपको अभी इस पेज में बताई हैं। ठीक है? सो सुपर एक्साइटेड फॉर दैट प्लेलिस्ट एज वेल। आई होप आपको थोड़ा सा एक ग्लिं्स मिला कि एडवांस्ड रैक सिस्टम्स क्यों एग्जिस्ट करते हैं और वहां पे क्या-क्या टेक्निक्स होती हैं। ठीक है? बाकी आई होप इस वीडियो के थ्रू मैं आपको समझा पाया एक सिंपल फंक्शनल रैक सिस्टम कैसे बनाया जाता है। अगर आपको वीडियो पसंद आया तो प्लीज लाइक करना। अगर आपने इस चैनल को सब्सक्राइब नहीं किया है, प्लीज डू सब्सक्राइब। मिलते हैं नेक्स्ट वीडियो में। बाय।')

1(c) & 1(d): Indexing(Embedding Geberation and Storing in Vector Store)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

embeddings.embed_query("Hello world")

[-0.02342152,
 0.01676572,
 0.009261323,
 -0.06383,
 -0.0026262768,
 0.0010187156,
 -0.01125684,
 0.0130036585,
 0.008751671,
 0.0012745215,
 -0.0007880878,
 -0.019086141,
 0.030971918,
 0.044264916,
 0.11137619,
 0.018025596,
 -0.0031383373,
 -0.0130702155,
 0.0054121013,
 -0.009036983,
 0.009904047,
 -0.010718052,
 0.012551899,
 0.011420718,
 -0.03375867,
 0.008877066,
 0.027795823,
 0.0037452083,
 0.029841818,
 0.019861836,
 0.018075233,
 -0.007178086,
 -0.02497957,
 0.011720823,
 -0.0073324037,
 0.0068515264,
 0.011618152,
 0.004278904,
 -0.0099426275,
 -0.009696086,
 -0.0037028084,
 0.006349137,
 -0.012098703,
 -0.015629271,
 -0.022799378,
 -0.012305125,
 -0.01497548,
 -0.006503783,
 -0.005624279,
 0.020265369,
 -0.029982245,
 -0.008486281,
 -0.0050998786,
 -0.14985937,
 -0.017222198,
 0.011674307,
 -0.014542328,
 0.02246327,
 -0.009126675,
 -0.018621653,
 -0.004780002,
 0.0020153068,
 -0.0133157885,
 -0.006439805,
 0.0008783784,
 -0.023549458,
 -0.008333907,
 0.01863065,
 -0.0190

In [ ]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id

{0: '6815cd2e-90a1-4889-807f-534965d76810',
 1: 'd5e64eec-8f9a-44ea-a503-2e5839b4685a',
 2: 'f9f7e0b4-f45d-40b1-8ae4-db3513f2e7fc',
 3: '44bc3d8c-123c-4b23-b13c-7f251bb5a582',
 4: '47b738be-f76c-4d3d-978f-d02f7c443732',
 5: '22252bb9-822a-4eb3-8883-07fa8a858785',
 6: 'a7c8781e-d506-43e0-9903-b69089e56b52',
 7: 'edbd20a8-6e5e-4f65-9eb2-99f44b0ca5f7',
 8: 'c1af4bfd-ae6b-459a-8c7c-dcd316e2523b',
 9: 'c3b01290-dac1-4834-bcaa-2cf300423c05',
 10: '2cab780c-0a55-4c61-b7c4-907d47162219',
 11: 'd1b0500c-545b-4bc0-a967-e9cd1e8391da',
 12: '0d83b41b-c1db-47ce-80c1-2f6461542861',
 13: 'b59bbfa7-fc68-4ba3-a817-5be022de3bec',
 14: '1d7d1ce6-a40e-449c-a176-7615a5c20dff',
 15: '8356886c-52a9-4838-b1d8-611f4fc0ea59',
 16: '77adda83-51f3-4704-aa94-3961d874de94',
 17: '6cf92ccb-4900-4728-b36e-4d1305166a20',
 18: 'd0072319-280f-497f-80cb-cde375606bf8',
 19: 'e65fb5e8-82ad-4faf-81ad-4b49d0e79d95',
 20: '6cb84d7b-024d-44d6-8839-13bdf263b2fb',
 21: '8286c549-8817-49d5-ab88-8d6be650ef7b',
 22: 'cbddc446-5493-

Step 2: Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7986b42f8650>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke('UI')

[Document(id='47b738be-f76c-4d3d-978f-d02f7c443732', metadata={}, page_content='की नॉलेज लगेगी। सेकंड ऑप्शन इज़ अगर आपको मान लो HTML, CSS, जावास्क्रिप्ट नहीं आता तो व्हाट यू कैन डू इज़ यू कैन मेक अ स्ट्रीमलेट वेबसाइट जहां पे आप यहां पर YouTube वीडियो का लिंक पेस्ट कर दोगे और जैसे ही आप सबमिट पे क्लिक करोगे एक नए विंडो में एक चैट विंडो ओपन हो जाएगी और यहां पे आप चैट कर सकते हो उस वीडियो के बारे में। तो इसके लिए आपको HTML, CSS, jवास्क्रिप्ट नहीं आना चाहिए बट आपको स्ट्रीमलेट जैसे टूल के साथ-साथ काम करना आना चाहिए। ठीक है? अ बट हम आज सिंस फोकस करने वाले हैं ऑन रैग तो मैं फिलहाल यूआई के ऊपर ज्यादा फोकस नहीं करूंगा। मैं फंक्शनैलिटी के ऊपर ज्यादा फोकस करूंगा। एंड दैट इज व्हाई हम जो भी बनाएंगे वो एक Google कोलैब नोटबुक के अंदर ही बनाएंगे। बट आई वुड रिकमेंड कि जब आप Google कोलैब में एक बार इस प्रोजेक्ट को चला के देख लोगे उसके बाद डेफिनेटली यू शुड ट्राई एंड बिल्ड अ यूआई अराउंड दिस प्रोजेक्ट। इससे क्या होगा? आपका प्रोजेक्ट थोड़ा और अच्छा दिखाई देगा। थोड़ा ज्यादा यूज़बल हो जाएगा। ठीक है? तो नटशेल में द

Step 3: Augmentatiom

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

In [ ]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}

Question: {question}
""",
    input_variables=["context", "question"]
)

In [ ]:
question = "is the topic of retriever discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'लाके आप तक देगा। ठीक है? सो व्हाट आई व्हाट आई विल डू इज़ कि सबसे पहले मैं एक रिट्रीवर फॉर्म कर रहा हूं। और हम एक बहुत सिंपल रिट्रीवर बना रहे हैं। हमारा जो वेक्टर स्टोर है, हम उसी को यूज़ करके हमारा रिट्रीवर बना रहे हैं। जिसका सर्च टाइप इज़ सिमिलॅरिटी सर्च। और यह आपको चार सबसे सिमिलर वेक्टर्स निकाल करके देगा, खोज करके देगा। ठीक है? सो इस कोड को हमने रन कर लिया। ये रहा हमारा रिट्रीवर। अब रिट्रीवर सिंस खुद में एक रननेबल होता है तो उसके पास इनवोक फंक्शन होता है। तो हम इनवोक फंक्शन को कॉल करेंगे। और यहां पे हम अपना क्वेरी पूछेंगे। सो हमारे केस में हमारा जो क्वेरी है वो है व्हाट इज डीप माइंड? बट आप कोई और क्वेश्चन भी पूछ सकते हो जो इस वीडियो से रिलेटेड है। सो हमने सर्च किया व्हाट इज डीप माइंड? तो इन टोटल जैसा मुझे दिखाई दे रहा है हमारे रिट्रीवर ने चार डॉक्यूमेंट्स दिए बिकॉज़ हमने उसे बता रखा है कि उसको चार डॉक्यूमेंट्स देने हैं। एंड हमारे रिट्रीवर को लगता है कि ये चार जो डॉक्यूमेंट्स हैं वो इस क्वेश्चन को आंसर करने के लिए सही हैं। ठीक है? तो बेसिकली हमारा जो रिट्रीवल वाला स्टेप है दैट इज़\n\nमें

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text="\nYou are a helpful assistant.\nAnswer ONLY from the provided transcript context.\nIf the context is insufficient, just say you don't know.\n\nलाके आप तक देगा। ठीक है? सो व्हाट आई व्हाट आई विल डू इज़ कि सबसे पहले मैं एक रिट्रीवर फॉर्म कर रहा हूं। और हम एक बहुत सिंपल रिट्रीवर बना रहे हैं। हमारा जो वेक्टर स्टोर है, हम उसी को यूज़ करके हमारा रिट्रीवर बना रहे हैं। जिसका सर्च टाइप इज़ सिमिलॅरिटी सर्च। और यह आपको चार सबसे सिमिलर वेक्टर्स निकाल करके देगा, खोज करके देगा। ठीक है? सो इस कोड को हमने रन कर लिया। ये रहा हमारा रिट्रीवर। अब रिट्रीवर सिंस खुद में एक रननेबल होता है तो उसके पास इनवोक फंक्शन होता है। तो हम इनवोक फंक्शन को कॉल करेंगे। और यहां पे हम अपना क्वेरी पूछेंगे। सो हमारे केस में हमारा जो क्वेरी है वो है व्हाट इज डीप माइंड? बट आप कोई और क्वेश्चन भी पूछ सकते हो जो इस वीडियो से रिलेटेड है। सो हमने सर्च किया व्हाट इज डीप माइंड? तो इन टोटल जैसा मुझे दिखाई दे रहा है हमारे रिट्रीवर ने चार डॉक्यूमेंट्स दिए बिकॉज़ हमने उसे बता रखा है कि उसको चार डॉक्यूमेंट्स देने हैं। एं

Step 4: Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

हाँ, रिट्रीवर के विषय में इस वीडियो में चर्चा की गई है।

जो चर्चा की गई है वह इस प्रकार है:
*   **रिट्रीवर का निर्माण:** एक रिट्रीवर को वेक्टर स्टोर का उपयोग करके बनाया जाता है।
*   **कार्यप्रणाली:**
    *   यह एक क्वेरी लेता है।
    *   यह क्वेरी को एंबेड करके वेक्टर के रूप में बदलता है।
    *   फिर यह वेक्टर स्टोर में जाकर उस दिए गए वेक्टर के सबसे करीब के वेक्टर्स को खोजता है।
    *   यह उन वेक्टर्स के अनुरूप चंक्स या डॉक्यूमेंट्स (जैसे चार सबसे समान डॉक्यूमेंट्स) को निकाल कर देता है।
*   **विशेषताएँ:**
    *   यह खुद में एक रननेबल होता है और इसके पास `invoke` फंक्शन होता है।
    *   उदाहरण में, इसका सर्च टाइप सिमिलैरिटी सर्च है और यह चार सबसे सिमिलर वेक्टर्स/डॉक्यूमेंट्स देता है।
    *   इसे रिट्रीवल वाले स्टेप में उपयोग किया जाता है।
*   **उन्नत उपयोग:**
    *   मल्टीपल रिट्रीवर्स के साथ डोमेन अवेयर राउटिंग की जा सकती है, जहाँ क्वेरी के आधार पर अलग-अलग रिट्रीवर ट्रिगर होते हैं।
    *   रिट्रीवल के दौरान MMR (मैक्सिमम मार्जिनल रेलेवेंस) जैसी सर्च स्ट्रेटेजी या हाइब्रिड रिट्रीवल (सिम

Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('How indexing is used in RAG pipeline')

{'context': 'था कि स्टेप वन होता है किसी भी रैग आर्किटेक्चर में। उसके बाद व्हाट वी विल डू इज कि वी विल क्रिएट अ रिट्रीवर। ठीक है? फिलहाल हम एक बहुत सिंपल सिमिलरिटी सर्च बेस्ड रिट्रीवर यूज़ करेंगे और हम उस रिट्रीवर को एक क्वेरी सेंड करेंगे। ठीक है? अब यह रिट्रीवर अपना काम करेगा। इस क्वेरी को एंबेड करेगा और जाकर के एक सिमेंटिक सर्च परफॉर्म करेगा वेक्टर स्टोर में और आपको रेलेवेंट डॉक्यूमेंट्स ला करके देगा। ठीक है? तो ये जो स्टेप है यह है आपका स्टेप नंबर टू रिट्रीवल। एक बार जैसे ही आपके पास रेलेवेंट चंक्स आ गए और आपके पास आपकी क्वेरी तो है ही। अब आप इन दोनों को मर्ज करके एक प्र्प्ट बनाओगे। दिस विल बी स्टेप नंबर थ्री जिसको हम आर्गुमेंटेशन बोलते हैं। और एक बार जैसे ही मेरे पास यह प्र्प्ट आ जाता है इस प्र्प्ट को मैं अपने एलएलएम के पास भेजूंगा और मेरा एलएलएम क्वेरी को समझेगा। इस कॉन्टेक्स्ट को समझेगा और पलट के मुझे एक रिस्पांस जनरेट करके देगा और यह होगा हमारा फोर्थ एंड फाइनल स्टेप दैट इज़ जनरेशन। ठीक है? तो, यह सारा काम पहले तो मैं आपको स्टेप बाय स्टेप करके दिखाऊंगा तोड़-तोड़ के। और फिर जब ये पू

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Summarize the video')

'यह वीडियो एक RAG (Retrieval Augmented Generation) आधारित सिस्टम के बारे में है जो उपयोगकर्ताओं को लंबे YouTube वीडियो को पूरा देखे बिना समझने में मदद करता है। यह सिस्टम उपयोगकर्ताओं को वीडियो के बारे में कोई भी प्रश्न पूछने की सुविधा देता है, जैसे कि क्या किसी विशेष विषय (जैसे AI, एलियंस, या न्यूक्लियर फ्यूजन) पर चर्चा हुई है, वीडियो को बुलेट पॉइंट्स में सारांशित करने, या लेक्चर के दौरान आने वाले संदेहों को हल करने के लिए।\n\nतकनीकी रूप से, यह सिस्टम YouTube API का उपयोग करके वीडियो ट्रांसक्रिप्ट लोड करता है। यह ट्रांसक्रिप्ट के टूटे हुए वाक्यों को एक साथ जोड़कर एक पूर्ण स्ट्रिंग बनाता है। फिर, यह उस ट्रांसक्रिप्ट को टेक्स्ट स्प्लिटर का उपयोग करके कई चंक्स में विभाजित करता है, उन चंक्स की एंबेडिंग्स जनरेट करता है, और उन्हें एक वेक्टर स्टोर में संग्रहीत करता है। अंत में, यह एक साधारण सिमिलरिटी सर्च-आधारित रिट्रीवर का उपयोग करके प्रश्नों का उत्तर देता है। यह मूल रूप से किसी भी YouTube वीडियो के बारे में कुछ भी पूछने के लिए एक चैट सिस्टम है।'